# Step 7 — Universal Bangla SyncNet

**Runtime → Change runtime type → A100 GPU, High-RAM.**

High-RAM is for the *vCPUs*, not the VRAM. Measured peak is ~2.5 GB at batch 32,
so an A100's 40 GB is never the constraint — feeding it is.

### Why this run exists

A linear probe recovers mouth aperture from AVE features at **r = 0.73** on Bangla
speakers the encoder never saw, so AVE is not discarding the information and
fine-tuning it would buy little. What collapses is transfer **between speakers**
(0.37), while transfer to a different *recording of the same person* does not
(0.79). The fault is speaker entanglement, and a multi-speaker lip-sync expert is
the direct fix. Details in `research/CONTEXT.md` §10.

### The rule for this notebook

A falling validation loss proves nothing on its own. The run is only believed if
the **offset curve peaks at 0 on held-out speakers** (last cell). A model can
separate pairs on some cue that is not lip-sync and still show a pretty loss
curve — that failure already cost this project one full training round.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, multiprocessing
print('vCPUs:', multiprocessing.cpu_count())
print('RAM GB:', round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9, 1))
!df -h /content | tail -1

## 1. Code and data

`research/` is gitignored, so `manifest.json` travels with the tars rather than
with the repo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# folder produced by research/corpus/pack_for_colab.py
UPLOAD = '/content/drive/MyDrive/fydp/colab_upload'
!ls -la {UPLOAD}

In [ ]:
%cd /content
!git clone https://github.com/YOUR_USER/YOUR_REPO.git repo 2>/dev/null || echo 'already cloned'
# if the repo is private, upload SyncTalk_2D as a zip instead and unzip here
!ls /content/repo/SyncTalk_2D/syncnet_corpus.py

In [ ]:
import time, os, glob
os.makedirs('/content/corpus', exist_ok=True)

# Copy to LOCAL disk first, then extract. Extracting straight from Drive, or
# worse training off it, turns 250k small files into hours of I/O.
t0 = time.time()
!cp {UPLOAD}/corpus_*.tar /content/
print(f'copied in {time.time()-t0:.0f}s')

t0 = time.time()
for t in sorted(glob.glob('/content/corpus_*.tar')):
    !tar -xf {t} -C /content/corpus
    print('extracted', os.path.basename(t))
!cp {UPLOAD}/manifest.json /content/manifest.json

# Crop-box caches ride along in the upload folder: ~4 MB that saves re-reading
# 250k landmark files to rebuild them (~9 min of A100 time doing file I/O).
!mkdir -p /content/boxcache && cp {UPLOAD}/boxcache/*.npy /content/boxcache/ 2>/dev/null
print(f'extracted in {time.time()-t0:.0f}s')

!rm -f /content/corpus_*.tar          # reclaim the space
print('videos:', len(os.listdir('/content/corpus')))
print('frames:', len(glob.glob('/content/corpus/*/full_body_img/*.jpg')))
print('box caches:', len(glob.glob('/content/boxcache/*.npy')), 'of 28')

## 2. Check the split before spending anything

`--dry_run` builds the datasets, prints the speaker-disjoint split and the step
count, and stops. If the validation speakers look wrong, fix it here rather than
two hours in.

In [ ]:
%cd /content/repo/SyncTalk_2D
!python syncnet_corpus.py --dry_run \
    --corpus /content/corpus --manifest /content/manifest.json \
    --cache_dir /content/boxcache --batch_size 32 --stride 2

## 3. Measure this machine before committing to it

Two numbers decide everything: how fast the **GPU** can step, and how fast the
**loader** can feed it. Whichever is smaller is your real throughput, and it sets
both the wall-clock and the bill. This takes about a minute and is the cheapest
cell in the notebook.

If the loader number is the smaller one, raise `NUM_WORKERS` and re-run this cell
before starting the real job.

In [ ]:
import sys, time, json, torch
sys.path.insert(0, '/content/repo/SyncTalk_2D')
from syncnet_328 import SyncNet_color, cosine_loss
from syncnet_corpus import CorpusDataset, split_speakers
from torch.utils.data import DataLoader

BATCH, STRIDE, NUM_WORKERS = 32, 2, 8
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# --- GPU side, synthetic input so only compute is timed -------------------
net = SyncNet_color('ave').cuda()
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
scaler = torch.cuda.amp.GradScaler()
img = torch.randn(BATCH, 3, 320, 320, device='cuda')
aud = torch.randn(BATCH, 32, 16, 16, device='cuda')
y = torch.ones(BATCH, 1, device='cuda')
def step():
    with torch.autocast('cuda', dtype=torch.float16):
        a, v = net(img, aud)
    loss = cosine_loss(a.float(), v.float(), y)   # BCELoss: fp32, outside autocast
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
for _ in range(8): step()
torch.cuda.synchronize(); t0 = time.time()
for _ in range(30): step()
torch.cuda.synchronize()
gpu_rate = BATCH * 30 / (time.time() - t0)
print(f'GPU     {gpu_rate:7.0f} samples/s   peak VRAM '
      f'{torch.cuda.max_memory_allocated()/1e9:.2f} GB')

# --- loader side ----------------------------------------------------------
man = json.load(open('/content/manifest.json'))
train_spk, _, _ = split_speakers(man, 4)
ds = CorpusDataset('/content/corpus', man, train_spk, 'ave', STRIDE,
                   cache_dir='/content/boxcache')
dl = DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS,
                pin_memory=True, drop_last=True, prefetch_factor=4)
it = iter(dl)
for _ in range(5): next(it)                      # let the workers spin up
t0 = time.time()
for _ in range(30): next(it)
load_rate = BATCH * 30 / (time.time() - t0)
print(f'loader  {load_rate:7.0f} samples/s   ({NUM_WORKERS} workers)')

rate = min(gpu_rate, load_rate)
ep_min = len(ds) / rate / 60
print(f'\nbottleneck: {"GPU" if gpu_rate < load_rate else "LOADER"}'
      f'  ->  {rate:.0f} samples/s')
print(f'{len(ds):,} samples/epoch  ->  {ep_min:.1f} min/epoch')
for e in (15, 20, 30):
    print(f'   {e:>2} epochs   {ep_min*e/60:>4.1f} h')
print('\nEarly stopping means you pay for roughly (best epoch + patience),')
print('not the full ceiling. Expect the peak around epoch 10-15.')
del net, opt, img, aud, y, it, dl, ds; torch.cuda.empty_cache()

## 4. Train

**`save_dir` points at Drive on purpose.** Checkpoints are written once an epoch,
so the write cost is nothing next to losing the run when a session dies. Re-running
this cell resumes from `last.pth` — epoch, optimizer, AMP scaler and the patience
counter all carry across.

### Settings, and why

| | |
|---|---|
| `--stride 2` | adjacent frames at 25 fps are near-duplicates. **Halves cost per epoch** for almost no loss of diversity |
| `--val_stride 8` | validation only needs to be a stable estimate, not exhaustive |
| `--batch_size 32` | with stride 2 this keeps ~3.3k steps/epoch. Batch 128 would drop it to ~830 and underfit |
| `--amp` | **1.57×**, measured |
| `--epochs 30 --patience 3` | 30 is a ceiling you probably never reach; it stops 3 epochs after the peak |

Measured and **rejected**: `channels_last` runs **5× slower** for this network
(28 vs 152 samples/s). TF32 is on and worth ~4%. Do not re-add channels_last
without measuring it again.

Capping epochs low does not save units if you guess wrong — you would resume
anyway and pay the same compute plus another session's setup.

In [ ]:
SAVE = '/content/drive/MyDrive/fydp/syncnet_ckpt/universal_bn'
import os
os.makedirs(SAVE, exist_ok=True)
RESUME = f'{SAVE}/last.pth' if os.path.exists(f'{SAVE}/last.pth') else ''
print('resuming from', RESUME or '(fresh start)')

!python syncnet_corpus.py \
    --corpus /content/corpus --manifest /content/manifest.json \
    --cache_dir /content/boxcache --save_dir {SAVE} \
    --epochs 30 --patience 3 --batch_size 32 \
    --stride 2 --val_stride 8 \
    --num_workers 8 --amp --resume "{RESUME}"

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv(f'{SAVE}/train_log.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(df.epoch, df.train_loss, label='train')
ax[0].plot(df.epoch, df.val_loss, label='val (held-out speakers)')
ax[0].axvline(df.epoch[df.val_loss.idxmin()], ls='--', c='k', lw=1)
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend()
ax[1].plot(df.epoch, df.gap, c='tab:green')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('pos_sim - neg_sim')
ax[1].set_title('separation on unseen voices')
plt.tight_layout(); plt.show()
print('best val at epoch', int(df.epoch[df.val_loss.idxmin()]), '=', df.val_loss.min())

## 5. The acceptance test

Shift the audio against the video and score true pairs only. It must peak at 0
**on held-out speakers**. Then run it again with `--on_train_speakers`: a sharp
curve on training voices next to a flat one on unseen voices is exactly the
speaker entanglement this corpus was built to remove, and would mean the run
failed at its actual job however good the loss looks.

In [ ]:
!python eval_sync_offset.py --ckpt {SAVE}/best_val.pth \
    --corpus /content/corpus --manifest /content/manifest.json \
    --cache_dir /content/boxcache
print('\n' + '='*70 + '\nSAME CHECKPOINT, TRAINING VOICES — for contrast\n' + '='*70)
!python eval_sync_offset.py --ckpt {SAVE}/best_val.pth \
    --corpus /content/corpus --manifest /content/manifest.json \
    --cache_dir /content/boxcache --on_train_speakers